# 03. Modeling — ARIMA, SARIMA, and SARIMAX Core Outputs

This notebook is the core modeling notebook for the final project. It fits ARIMA, SARIMA, SARIMAX, benchmark, rolling-origin, intervention, diagnostics, and counterfactual models, then exports **raw reproducible outputs** for notebooks 04–06.

Pipeline rule: this notebook should not depend on files created by notebooks 04, 05, or 06.

## Pipeline role of Notebook 03

Notebook 03 is intentionally the upstream producer. It writes raw model outputs with prefixes such as `anxiety_*` and `final_anxiety_*`. Notebook 04 reads those outputs for forecast evaluation. Notebook 05 reads those outputs for SARIMAX intervention and counterfactual analysis. Notebook 06 reads 04 and 05 outputs for the final cross-national comparison.

This avoids circular dependencies where a notebook reads its own final output before creating it.

## A. Modeling Design

This notebook separates three tasks:

| Task | Main models | Purpose |
|---|---|---|
| Forecast evaluation | Naive, Seasonal Naive, ARIMA, SARIMA, SARIMAX | Compare out-of-sample predictive performance |
| COVID intervention analysis | SARIMAX full-sample intervention | Estimate COVID-period and post-COVID level/trend effects |
| Counterfactual analysis | Pre-COVID SARIMA | Estimate no-COVID baseline and observed-vs-counterfactual gaps |

### COVID Period Definition

The COVID phase structure is aligned with `01_Preprocessing.ipynb` and `02_EDA.ipynb`:

- **Pre-COVID:** 2004-01 to 2019-12
- **COVID period:** 2020-01 to 2023-05
- **Post-COVID:** 2023-06 to 2025-12

### Forecast Split

- **Train:** 2004-01 to 2021-12
- **Validation:** 2022-01 to 2023-05
- **Test:** 2023-06 to 2025-12

This split allows the model to learn early/mid COVID behavior before validation and reserves the post-COVID period for final test evaluation.

In [23]:
from pathlib import Path
from itertools import product
import ast
import warnings
import zipfile

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display

from scipy import stats
from scipy.special import logit, expit
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.stats.diagnostic import acorr_ljungbox, het_arch
from statsmodels.graphics.tsaplots import plot_acf

warnings.filterwarnings("ignore")

PROJECT_ROOT = Path.cwd()
DATA_PATH = Path("data/01_input_modeling.csv")
RESULTS_DIR = Path("modeling_outputs")
FIGURES_DIR = Path("figures")

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

print("Current working directory:", PROJECT_ROOT)
print("Data path:", DATA_PATH)
print("File exists:", DATA_PATH.exists())
print("Results directory:", RESULTS_DIR)
print("Figures directory:", FIGURES_DIR)

Current working directory: c:\Mental search trend final
Data path: data\01_input_modeling.csv
File exists: True
Results directory: modeling_outputs
Figures directory: figures


## B. Load Modeling Dataset

This notebook uses `input_modeling.csv`, created by preprocessing. It must contain:

- the five Google Trends variables;
- country and monthly date identifiers;
- COVID intervention variables used in SARIMAX.

If the file is missing, run `01_Preprocessing.ipynb` first.

In [24]:
if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Cannot find {DATA_PATH}. Run 01_Preprocessing.ipynb first and make sure the project root is the working directory."
    )

df = pd.read_csv(DATA_PATH)
df["date"] = pd.to_datetime(df["date"])
df = df.sort_values(["country", "date"]).reset_index(drop=True)

TARGET_VAR = "anxiety"
MENTAL_VARS = ["anxiety", "depression", "stress", "insomnia", "mental_disorder"]
COUNTRIES = sorted(df["country"].unique())

print("Shape:", df.shape)
print("Date range:", df["date"].min(), "→", df["date"].max())
print("Countries:", COUNTRIES)
print("Total missing values:", int(df.isna().sum().sum()))

display(df.head())

Shape: (1584, 20)
Date range: 2004-01-01 00:00:00 → 2025-12-01 00:00:00
Countries: ['Australia', 'Canada', 'Ireland', 'New_Zealand', 'UK', 'US']
Total missing values: 0


,date,country,anxiety,depression,stress,insomnia,mental_disorder,anxiety_was_imputed,depression_was_imputed,stress_was_imputed,insomnia_was_imputed,mental_disorder_was_imputed,year,month,time_index,analysis_phase,covid_period,covid_trend,post_covid_period,post_covid_trend
0,2004-01-01,Australia,27.0,55.0,28.0,6.0,16.0,0,0,0,0,0,2004,1,0,pre_covid,0,0,0,0
1,2004-02-01,Australia,31.0,63.0,34.0,9.0,28.0,0,0,0,0,0,2004,2,1,pre_covid,0,0,0,0
2,2004-03-01,Australia,32.0,82.0,45.0,9.0,40.0,0,0,0,0,0,2004,3,2,pre_covid,0,0,0,0
3,2004-04-01,Australia,33.0,73.0,45.0,5.0,34.0,0,0,0,0,0,2004,4,3,pre_covid,0,0,0,0
4,2004-05-01,Australia,33.0,87.0,49.0,8.0,38.0,0,0,0,0,0,2004,5,4,pre_covid,0,0,0,0


## C. Shared COVID Configuration and Additional Intervention Variable

The base preprocessing already defines:

- `covid_period`
- `covid_trend`
- `post_covid_period`
- `post_covid_trend`

This notebook adds one extra intervention variable:

- `early_covid_shock`: March 2020–June 2020

Reason: COVID effects may have been strongest during the initial shock period rather than evenly distributed over the full 2020-01 to 2023-05 COVID period. This avoids forcing a single long COVID dummy to explain both the first pandemic shock and later adaptation phases.

In [25]:
# C. MODELING CONFIGURATION

COVID_START = pd.Timestamp("2020-01-01")
COVID_END = pd.Timestamp("2023-05-01")
POST_COVID_START = pd.Timestamp("2023-06-01")
PRE_COVID_END = pd.Timestamp("2019-12-01")

EARLY_COVID_START = pd.Timestamp("2020-03-01")
EARLY_COVID_END = pd.Timestamp("2020-06-01")

TRAIN_END = pd.Timestamp("2021-12-01")
VAL_START = pd.Timestamp("2022-01-01")
VAL_END = pd.Timestamp("2023-05-01")
TEST_START = pd.Timestamp("2023-06-01")

# Create variables defensively in case input_modeling.csv was produced by an older preprocessing file.
df["covid_period"] = ((df["date"] >= COVID_START) & (df["date"] <= COVID_END)).astype(int)
df["post_covid_period"] = (df["date"] >= POST_COVID_START).astype(int)
df["early_covid_shock"] = ((df["date"] >= EARLY_COVID_START) & (df["date"] <= EARLY_COVID_END)).astype(int)

# Recreate trend counters within each country for consistency.
df["covid_trend"] = 0
df["post_covid_trend"] = 0

for country in COUNTRIES:
    m_country = df["country"].eq(country)
    m_covid = m_country & (df["date"] >= COVID_START) & (df["date"] <= COVID_END)
    m_post = m_country & (df["date"] >= POST_COVID_START)
    df.loc[m_covid, "covid_trend"] = np.arange(1, int(m_covid.sum()) + 1)
    df.loc[m_post, "post_covid_trend"] = np.arange(1, int(m_post.sum()) + 1)

# Forecast SARIMAX should use only variables observable/learnable before the post-COVID test period.
# Post-COVID coefficients are estimated in full-sample explanatory SARIMAX, not in prospective test forecasting.
EXOG_FORECAST = ["early_covid_shock", "covid_period", "covid_trend"]

# Full-sample explanatory SARIMAX uses all intervention terms.
EXOG_EXPLANATORY = [
    "early_covid_shock",
    "covid_period",
    "covid_trend",
    "post_covid_period",
    "post_covid_trend",
]

print("COVID configuration:")
print("Pre-COVID ends:", PRE_COVID_END.date())
print("COVID period:", COVID_START.date(), "→", COVID_END.date())
print("Early COVID shock:", EARLY_COVID_START.date(), "→", EARLY_COVID_END.date())
print("Post-COVID starts:", POST_COVID_START.date())
print("Train ends:", TRAIN_END.date())
print("Validation:", VAL_START.date(), "→", VAL_END.date())
print("Test starts:", TEST_START.date())
print("Forecast exog:", EXOG_FORECAST)
print("Explanatory exog:", EXOG_EXPLANATORY)

COVID configuration:
Pre-COVID ends: 2019-12-01
COVID period: 2020-01-01 → 2023-05-01
Early COVID shock: 2020-03-01 → 2020-06-01
Post-COVID starts: 2023-06-01
Train ends: 2021-12-01
Validation: 2022-01-01 → 2023-05-01
Test starts: 2023-06-01
Forecast exog: ['early_covid_shock', 'covid_period', 'covid_trend']
Explanatory exog: ['early_covid_shock', 'covid_period', 'covid_trend', 'post_covid_period', 'post_covid_trend']


## D. Validate Train / Validation / Test Split

This check prevents a common SARIMAX mistake: fitting COVID intervention models on a training period that contains no COVID observations.

Expected structure per country:

- Train contains pre-COVID + early COVID observations.
- Validation is late COVID.
- Test is post-COVID.

In [26]:
def assign_split(date):
    if date <= TRAIN_END:
        return "train"
    if VAL_START <= date <= VAL_END:
        return "validation"
    if date >= TEST_START:
        return "test"
    return "gap"

split_df = df.copy()
split_df["split"] = split_df["date"].apply(assign_split)

split_check = (
    split_df.groupby(["country", "split"])
    .agg(
        start_date=("date", "min"),
        end_date=("date", "max"),
        n_months=("date", "count"),
        covid_months=("covid_period", "sum"),
        early_covid_months=("early_covid_shock", "sum"),
        post_covid_months=("post_covid_period", "sum"),
        max_covid_trend=("covid_trend", "max"),
        max_post_covid_trend=("post_covid_trend", "max"),
    )
    .reset_index()
)

display(split_check)

if (split_check.loc[split_check["split"].eq("train"), "covid_months"] == 0).any():
    raise ValueError("Train split contains no COVID observations for at least one country. SARIMAX intervention cannot be learned.")

if "gap" in split_check["split"].unique():
    raise ValueError("Split has an unintended gap. Check TRAIN_END, VAL_START, VAL_END, and TEST_START.")

split_check.to_csv(RESULTS_DIR / "03_anxiety_split_verification_enhanced.csv", index=False)

,country,split,start_date,end_date,n_months,covid_months,early_covid_months,post_covid_months,max_covid_trend,max_post_covid_trend
0,Australia,test,2023-06-01,2025-12-01,31,0,0,31,0,31
1,Australia,train,2004-01-01,2021-12-01,216,24,4,0,24,0
2,Australia,validation,2022-01-01,2023-05-01,17,17,0,0,41,0
3,Canada,test,2023-06-01,2025-12-01,31,0,0,31,0,31
4,Canada,train,2004-01-01,2021-12-01,216,24,4,0,24,0
5,Canada,validation,2022-01-01,2023-05-01,17,17,0,0,41,0
6,Ireland,test,2023-06-01,2025-12-01,31,0,0,31,0,31
7,Ireland,train,2004-01-01,2021-12-01,216,24,4,0,24,0
8,Ireland,validation,2022-01-01,2023-05-01,17,17,0,0,41,0
9,New_Zealand,test,2023-06-01,2025-12-01,31,0,0,31,0,31


## E. Candidate Orders Derived from EDA

The candidate set follows the EDA logic:

| EDA evidence | Modeling implication |
|---|---|
| ADF/KPSS: level mostly non-stationary, first difference more stable | set `d = 1` |
| ACF spike near lag 1 | include `ARIMA(0,1,1)` |
| ACF + PACF short-lag signals | include `ARIMA(1,1,1)` |
| lag 1–2/noisy dependence | include `ARIMA(2,1,1)` and `ARIMA(1,1,2)` |
| seasonal spikes at 12/24/36 | use `s = 12` in SARIMA |
| uncertain seasonal differencing | compare `D = 0` and `D = 1` |
| possible seasonal AR/MA behavior | test seasonal AR and seasonal MA terms |

The candidate set is deliberately moderate. A huge grid would overfit a 264-month time series.

In [27]:
ARIMA_ORDERS = [
    (0, 1, 1),
    (1, 1, 1),
    (2, 1, 1),
    (1, 1, 2),
    (2, 1, 2),
    (3, 1, 1),
    (1, 1, 3),
]

SARIMA_SPECS = [
    # Seasonal MA without seasonal differencing
    ((0, 1, 1), (0, 0, 1, 12)),
    ((1, 1, 1), (0, 0, 1, 12)),
    ((2, 1, 1), (0, 0, 1, 12)),
    ((1, 1, 2), (0, 0, 1, 12)),

    # Seasonal AR without seasonal differencing
    ((0, 1, 1), (1, 0, 0, 12)),
    ((1, 1, 1), (1, 0, 0, 12)),
    ((2, 1, 1), (1, 0, 0, 12)),

    # Seasonal MA with seasonal differencing
    ((0, 1, 1), (0, 1, 1, 12)),
    ((1, 1, 1), (0, 1, 1, 12)),
    ((2, 1, 1), (0, 1, 1, 12)),
    ((1, 1, 2), (0, 1, 1, 12)),

    # Seasonal AR with seasonal differencing
    ((0, 1, 1), (1, 1, 0, 12)),
    ((1, 1, 1), (1, 1, 0, 12)),
    ((2, 1, 1), (1, 1, 0, 12)),

    # Richer seasonal candidates used only if they pass validation/diagnostics
    ((1, 1, 1), (1, 0, 1, 12)),
    ((1, 1, 1), (1, 1, 1, 12)),
]

SARIMAX_SPECS = SARIMA_SPECS.copy()

print("Number of ARIMA candidates:", len(ARIMA_ORDERS))
print("Number of SARIMA candidates:", len(SARIMA_SPECS))
print("Number of SARIMAX candidates:", len(SARIMAX_SPECS))

Number of ARIMA candidates: 7
Number of SARIMA candidates: 16
Number of SARIMAX candidates: 16


## F. Utility Functions

This section defines reusable functions for:

- extracting country-level time series;
- fitting ARIMA/SARIMA/SARIMAX models;
- computing forecast errors;
- running residual diagnostics;
- running Diebold-Mariano forecast comparison tests.

In [28]:
def get_country_frame(data, country):
    out = data.loc[data["country"].eq(country)].copy()
    out = out.sort_values("date").set_index("date")
    out = out.asfreq("MS")
    return out


def get_series(data, country, variable=TARGET_VAR):
    return get_country_frame(data, country)[variable].astype(float)


def get_exog(data, country, cols):
    return get_country_frame(data, country)[cols].astype(float)


def smape(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    denom = (np.abs(y_true) + np.abs(y_pred)) / 2
    mask = denom != 0
    if mask.sum() == 0:
        return np.nan
    return np.mean(np.abs(y_true[mask] - y_pred[mask]) / denom[mask]) * 100


def forecast_metrics(y_true, y_pred):
    y_true = pd.Series(y_true).astype(float)
    y_pred = pd.Series(y_pred, index=y_true.index).astype(float)
    err = y_true - y_pred
    return {
        "MAE": float(np.mean(np.abs(err))),
        "RMSE": float(np.sqrt(np.mean(err ** 2))),
        "sMAPE": float(smape(y_true, y_pred)),
    }


def fit_sarimax_model(y, order, seasonal_order, exog=None, maxiter=300):
    model = SARIMAX(
        y,
        exog=exog,
        order=order,
        seasonal_order=seasonal_order,
        trend="c",
        enforce_stationarity=False,
        enforce_invertibility=False,
    )
    return model.fit(disp=False, maxiter=maxiter)


def forecast_from_fit(fit, steps, index, exog=None):
    pred = fit.get_forecast(steps=steps, exog=exog)
    mean = pred.predicted_mean
    mean.index = index
    return mean


def forecast_with_ci(fit, steps, index, exog=None, alpha=0.05):
    pred = fit.get_forecast(steps=steps, exog=exog)
    mean = pred.predicted_mean
    ci = pred.conf_int(alpha=alpha)
    mean.index = index
    ci.index = index
    return mean, ci


def safe_ljungbox_p(resid, lag):
    resid = pd.Series(resid).dropna()
    if len(resid) <= lag + 2:
        return np.nan
    return float(acorr_ljungbox(resid, lags=[lag], return_df=True)["lb_pvalue"].iloc[0])


def safe_arch_lm_p(resid, lags=12):
    resid = pd.Series(resid).dropna()
    if len(resid) <= lags + 2:
        return np.nan
    try:
        return float(het_arch(resid, nlags=lags)[1])
    except Exception:
        return np.nan


def safe_jarque_bera_p(resid):
    resid = pd.Series(resid).dropna()
    if len(resid) < 8:
        return np.nan
    try:
        return float(stats.jarque_bera(resid).pvalue)
    except Exception:
        return np.nan


def residual_diagnostics(resid):
    resid = pd.Series(resid).dropna()
    return {
        "resid_n": int(len(resid)),
        "resid_mean": float(resid.mean()) if len(resid) else np.nan,
        "resid_std": float(resid.std()) if len(resid) else np.nan,
        "LjungBox_p12": safe_ljungbox_p(resid, 12),
        "LjungBox_p24": safe_ljungbox_p(resid, 24),
        "ARCH_LM_p12": safe_arch_lm_p(resid, 12),
        "JarqueBera_p": safe_jarque_bera_p(resid),
    }


def naive_forecast(y_train, y_future_index):
    last_value = float(y_train.iloc[-1])
    return pd.Series(last_value, index=y_future_index)


def seasonal_naive_forecast(y_train, y_future_index, season=12):
    preds = []
    y_hist = y_train.copy()
    for dt in y_future_index:
        ref_dt = dt - pd.DateOffset(months=season)
        if ref_dt in y_hist.index:
            val = float(y_hist.loc[ref_dt])
        else:
            val = float(y_hist.iloc[-1])
        preds.append(val)
        y_hist.loc[dt] = val
    return pd.Series(preds, index=y_future_index)


def diebold_mariano_test(e_model, e_benchmark, loss="squared", max_lag=12):
    # H0: equal predictive accuracy. Negative DM statistic means model has lower loss than benchmark.
    e_model = np.asarray(e_model, dtype=float)
    e_benchmark = np.asarray(e_benchmark, dtype=float)
    mask = np.isfinite(e_model) & np.isfinite(e_benchmark)
    e_model = e_model[mask]
    e_benchmark = e_benchmark[mask]
    n = len(e_model)
    if n < 5:
        return {"DM_stat": np.nan, "DM_pvalue": np.nan, "mean_loss_diff": np.nan, "n": n}

    if loss == "absolute":
        d = np.abs(e_model) - np.abs(e_benchmark)
    else:
        d = e_model ** 2 - e_benchmark ** 2

    dbar = float(np.mean(d))
    d_centered = d - dbar
    lag = min(max_lag, n - 1)

    gamma0 = np.mean(d_centered * d_centered)
    long_run_var = gamma0
    for k in range(1, lag + 1):
        gamma_k = np.mean(d_centered[k:] * d_centered[:-k])
        weight = 1 - k / (lag + 1)
        long_run_var += 2 * weight * gamma_k

    if long_run_var <= 0 or not np.isfinite(long_run_var):
        return {"DM_stat": np.nan, "DM_pvalue": np.nan, "mean_loss_diff": dbar, "n": n}

    dm_stat = dbar / np.sqrt(long_run_var / n)
    pvalue = 2 * (1 - stats.norm.cdf(abs(dm_stat)))
    return {
        "DM_stat": float(dm_stat),
        "DM_pvalue": float(pvalue),
        "mean_loss_diff": dbar,
        "n": int(n),
    }


def tuple_to_str(x):
    return str(tuple(x))


def parse_tuple(text):
    if isinstance(text, tuple):
        return text
    return tuple(ast.literal_eval(str(text)))

## G. Validation Model Selection

Models are fit on the training sample and evaluated on the validation sample.

Validation is used to select the best specification within each model class:

- Naive
- Seasonal Naive
- ARIMA
- SARIMA
- SARIMAX

SARIMAX is selected independently rather than borrowing the best SARIMA order. This is important because intervention variables can change the best error structure.

In [29]:
validation_rows = []
validation_forecasts = []
validation_fits = {}

for country in COUNTRIES:
    print(f"Validation selection: {country}")
    y = get_series(df, country, TARGET_VAR)
    x_forecast = get_exog(df, country, EXOG_FORECAST)

    y_train = y.loc[y.index <= TRAIN_END].dropna()
    y_val = y.loc[(y.index >= VAL_START) & (y.index <= VAL_END)].dropna()

    x_train = x_forecast.loc[y_train.index]
    x_val = x_forecast.loc[y_val.index]

    # Benchmarks
    benchmark_predictions = {
        "Naive": naive_forecast(y_train, y_val.index),
        "Seasonal_Naive": seasonal_naive_forecast(y_train, y_val.index, season=12),
    }
    for model_type, pred in benchmark_predictions.items():
        metrics = forecast_metrics(y_val, pred)
        validation_rows.append({
            "country": country,
            "model_type": model_type,
            "order": None,
            "seasonal_order": None,
            "exog_set": None,
            "AIC": np.nan,
            "BIC": np.nan,
            "converged": True,
            **{f"val_{k}": v for k, v in metrics.items()},
            "status": "OK",
        })
        for dt, actual, forecast in zip(y_val.index, y_val.values, pred.values):
            validation_forecasts.append({
                "country": country,
                "date": dt,
                "model_type": model_type,
                "order": None,
                "seasonal_order": None,
                "actual": actual,
                "forecast": forecast,
                "error": actual - forecast,
            })

    # ARIMA candidates
    for order in ARIMA_ORDERS:
        seasonal_order = (0, 0, 0, 0)
        try:
            fit = fit_sarimax_model(y_train, order, seasonal_order, exog=None)
            pred = forecast_from_fit(fit, len(y_val), y_val.index, exog=None)
            metrics = forecast_metrics(y_val, pred)
            validation_rows.append({
                "country": country,
                "model_type": "ARIMA",
                "order": tuple_to_str(order),
                "seasonal_order": tuple_to_str(seasonal_order),
                "exog_set": None,
                "AIC": float(fit.aic),
                "BIC": float(fit.bic),
                "converged": bool(fit.mle_retvals.get("converged", False)),
                **{f"val_{k}": v for k, v in metrics.items()},
                "status": "OK",
            })
            for dt, actual, forecast in zip(y_val.index, y_val.values, pred.values):
                validation_forecasts.append({
                    "country": country, "date": dt, "model_type": "ARIMA",
                    "order": tuple_to_str(order), "seasonal_order": tuple_to_str(seasonal_order),
                    "actual": actual, "forecast": forecast, "error": actual - forecast,
                })
        except Exception as e:
            validation_rows.append({
                "country": country, "model_type": "ARIMA", "order": tuple_to_str(order),
                "seasonal_order": tuple_to_str(seasonal_order), "exog_set": None,
                "AIC": np.nan, "BIC": np.nan, "converged": False,
                "val_MAE": np.nan, "val_RMSE": np.nan, "val_sMAPE": np.nan,
                "status": f"FAILED: {type(e).__name__}: {e}",
            })

    # SARIMA candidates
    for order, seasonal_order in SARIMA_SPECS:
        try:
            fit = fit_sarimax_model(y_train, order, seasonal_order, exog=None)
            pred = forecast_from_fit(fit, len(y_val), y_val.index, exog=None)
            metrics = forecast_metrics(y_val, pred)
            validation_rows.append({
                "country": country,
                "model_type": "SARIMA",
                "order": tuple_to_str(order),
                "seasonal_order": tuple_to_str(seasonal_order),
                "exog_set": None,
                "AIC": float(fit.aic),
                "BIC": float(fit.bic),
                "converged": bool(fit.mle_retvals.get("converged", False)),
                **{f"val_{k}": v for k, v in metrics.items()},
                "status": "OK",
            })
            for dt, actual, forecast in zip(y_val.index, y_val.values, pred.values):
                validation_forecasts.append({
                    "country": country, "date": dt, "model_type": "SARIMA",
                    "order": tuple_to_str(order), "seasonal_order": tuple_to_str(seasonal_order),
                    "actual": actual, "forecast": forecast, "error": actual - forecast,
                })
        except Exception as e:
            validation_rows.append({
                "country": country, "model_type": "SARIMA", "order": tuple_to_str(order),
                "seasonal_order": tuple_to_str(seasonal_order), "exog_set": None,
                "AIC": np.nan, "BIC": np.nan, "converged": False,
                "val_MAE": np.nan, "val_RMSE": np.nan, "val_sMAPE": np.nan,
                "status": f"FAILED: {type(e).__name__}: {e}",
            })

    # SARIMAX candidates with early-shock and COVID trend terms
    for order, seasonal_order in SARIMAX_SPECS:
        try:
            fit = fit_sarimax_model(y_train, order, seasonal_order, exog=x_train)
            pred = forecast_from_fit(fit, len(y_val), y_val.index, exog=x_val)
            metrics = forecast_metrics(y_val, pred)
            validation_rows.append({
                "country": country,
                "model_type": "SARIMAX",
                "order": tuple_to_str(order),
                "seasonal_order": tuple_to_str(seasonal_order),
                "exog_set": "+".join(EXOG_FORECAST),
                "AIC": float(fit.aic),
                "BIC": float(fit.bic),
                "converged": bool(fit.mle_retvals.get("converged", False)),
                **{f"val_{k}": v for k, v in metrics.items()},
                "status": "OK",
            })
            for dt, actual, forecast in zip(y_val.index, y_val.values, pred.values):
                validation_forecasts.append({
                    "country": country, "date": dt, "model_type": "SARIMAX",
                    "order": tuple_to_str(order), "seasonal_order": tuple_to_str(seasonal_order),
                    "actual": actual, "forecast": forecast, "error": actual - forecast,
                })
        except Exception as e:
            validation_rows.append({
                "country": country, "model_type": "SARIMAX", "order": tuple_to_str(order),
                "seasonal_order": tuple_to_str(seasonal_order), "exog_set": "+".join(EXOG_FORECAST),
                "AIC": np.nan, "BIC": np.nan, "converged": False,
                "val_MAE": np.nan, "val_RMSE": np.nan, "val_sMAPE": np.nan,
                "status": f"FAILED: {type(e).__name__}: {e}",
            })

validation_results = pd.DataFrame(validation_rows)
validation_forecasts_df = pd.DataFrame(validation_forecasts)

validation_results.to_csv(RESULTS_DIR / "03_anxiety_validation_model_selection_enhanced.csv", index=False)
validation_forecasts_df.to_csv(RESULTS_DIR / "03_anxiety_validation_forecasts_enhanced.csv", index=False)

display(validation_results.sort_values(["country", "val_RMSE"]).groupby("country").head(8))

Validation selection: Australia


Validation selection: Canada
Validation selection: Ireland
Validation selection: New_Zealand
Validation selection: UK
Validation selection: US


,country,model_type,order,seasonal_order,exog_set,AIC,BIC,converged,val_MAE,val_RMSE,val_sMAPE,status
24,Australia,SARIMA,"(1, 1, 1)","(1, 1, 1, 12)",None,1014.937656,1034.388138,True,4.127973,4.917401,4.839946,OK
21,Australia,SARIMA,"(1, 1, 1)","(1, 1, 0, 12)",None,1031.411367,1047.646487,True,4.284626,5.077824,4.991828,OK
22,Australia,SARIMA,"(2, 1, 1)","(1, 1, 0, 12)",None,1027.738410,1047.188893,True,4.272570,5.170491,4.986026,OK
20,Australia,SARIMA,"(0, 1, 1)","(1, 1, 0, 12)",None,1038.627711,1051.636805,True,4.393220,5.322115,5.129920,OK
1,Australia,Seasonal_Naive,None,None,None,NaN,NaN,True,4.705882,5.455704,5.426050,OK
33,Australia,SARIMAX,"(1, 1, 1)","(0, 1, 1, 12)",early_covid_shock+covid_period+covid_trend,1028.241158,1054.175134,True,4.491656,5.495307,5.290223,OK
34,Australia,SARIMAX,"(2, 1, 1)","(0, 1, 1, 12)",early_covid_shock+covid_period+covid_trend,1030.237297,1059.413021,True,4.491585,5.495861,5.290159,OK
40,Australia,SARIMAX,"(1, 1, 1)","(1, 1, 1, 12)",early_covid_shock+covid_period+covid_trend,1019.032765,1048.208488,True,4.489747,5.613202,5.287738,OK
41,Canada,Naive,None,None,None,NaN,NaN,True,5.235294,6.494341,5.978059,OK
51,Canada,SARIMA,"(1, 1, 1)","(0, 0, 1, 12)",None,1083.706103,1100.222627,True,5.878436,7.229150,6.682545,OK


## H. Select Best Model by Class

The best model within each class is selected using validation RMSE. AIC/BIC are secondary tie-breakers.

This preserves the methodological distinction between:

- best ARIMA;
- best SARIMA;
- best SARIMAX;
- best benchmark.

In [30]:
valid = validation_results.loc[validation_results["status"].eq("OK")].copy()
valid = valid.replace([np.inf, -np.inf], np.nan).dropna(subset=["val_RMSE"])

best_by_class = (
    valid.sort_values(["country", "model_type", "val_RMSE", "AIC", "BIC"], na_position="last")
    .groupby(["country", "model_type"], group_keys=False)
    .head(1)
    .reset_index(drop=True)
)

best_overall = (
    valid.sort_values(["country", "val_RMSE", "AIC", "BIC"], na_position="last")
    .groupby("country", group_keys=False)
    .head(1)
    .reset_index(drop=True)
)

best_by_class.to_csv(RESULTS_DIR / "03_anxiety_best_by_model_class_validation_enhanced.csv", index=False)
best_overall.to_csv(RESULTS_DIR / "03_anxiety_best_overall_validation_enhanced.csv", index=False)

display(best_by_class.sort_values(["country", "val_RMSE"]))
display(best_overall)

,country,model_type,order,seasonal_order,exog_set,AIC,BIC,converged,val_MAE,val_RMSE,val_sMAPE,status
2,Australia,SARIMA,"(1, 1, 1)","(1, 1, 1, 12)",None,1014.937656,1034.388138,True,4.127973,4.917401,4.839946,OK
4,Australia,Seasonal_Naive,None,None,None,NaN,NaN,True,4.705882,5.455704,5.426050,OK
3,Australia,SARIMAX,"(1, 1, 1)","(0, 1, 1, 12)",early_covid_shock+covid_period+covid_trend,1028.241158,1054.175134,True,4.491656,5.495307,5.290223,OK
0,Australia,ARIMA,"(3, 1, 1)","(0, 0, 0, 0)",None,1278.989489,1299.129007,True,5.563861,7.559817,6.526856,OK
1,Australia,Naive,None,None,None,NaN,NaN,True,16.882353,18.248288,21.190130,OK
6,Canada,Naive,None,None,None,NaN,NaN,True,5.235294,6.494341,5.978059,OK
7,Canada,SARIMA,"(1, 1, 1)","(0, 0, 1, 12)",None,1083.706103,1100.222627,True,5.878436,7.229150,6.682545,OK
9,Canada,Seasonal_Naive,None,None,None,NaN,NaN,True,6.058824,7.312359,6.805561,OK
5,Canada,ARIMA,"(2, 1, 1)","(0, 0, 0, 0)",None,1227.429371,1244.235832,True,6.404769,7.453934,7.267496,OK
8,Canada,SARIMAX,"(1, 1, 1)","(0, 0, 1, 12)",early_covid_shock+covid_period+covid_trend,1078.334379,1104.760819,True,6.595812,7.655416,7.461447,OK


,country,model_type,order,seasonal_order,exog_set,AIC,BIC,converged,val_MAE,val_RMSE,val_sMAPE,status
0,Australia,SARIMA,"(1, 1, 1)","(1, 1, 1, 12)",None,1014.937656,1034.388138,True,4.127973,4.917401,4.839946,OK
1,Canada,Naive,None,None,None,NaN,NaN,True,5.235294,6.494341,5.978059,OK
2,Ireland,Naive,None,None,None,NaN,NaN,True,4.529412,5.504009,5.660042,OK
3,New_Zealand,SARIMAX,"(0, 1, 1)","(1, 0, 0, 12)",early_covid_shock+covid_period+covid_trend,1232.971501,1256.163943,True,4.556762,6.195784,5.324248,OK
4,UK,Naive,None,None,None,NaN,NaN,True,5.941176,7.033533,6.857573,OK
5,US,SARIMAX,"(1, 1, 1)","(1, 1, 1, 12)",early_covid_shock+covid_period+covid_trend,884.173024,913.348748,True,3.634134,4.263065,4.186094,OK


## I. Test Evaluation of Selected Models

Selected models are refit on **train + validation** and evaluated on the post-COVID test period.

This is the final out-of-sample forecast check:

- Train + Validation: 2004-01 to 2023-05
- Test: 2023-06 to 2025-12

Important note: SARIMAX forecasting uses `early_covid_shock`, `covid_period`, and `covid_trend`. Post-COVID level/trend coefficients cannot be learned before the post-COVID test period, so post-COVID explanatory terms are reserved for full-sample intervention analysis.

In [31]:
test_rows = []
test_forecasts = []
selected_refit_residuals = {}
selected_fits = {}

for country in COUNTRIES:
    print(f"Test evaluation: {country}")
    y = get_series(df, country, TARGET_VAR)
    x_forecast = get_exog(df, country, EXOG_FORECAST)

    y_trainval = y.loc[y.index < TEST_START].dropna()
    y_test = y.loc[y.index >= TEST_START].dropna()
    x_trainval = x_forecast.loc[y_trainval.index]
    x_test = x_forecast.loc[y_test.index]

    country_best = best_by_class.loc[best_by_class["country"].eq(country)].copy()

    for _, row in country_best.iterrows():
        model_type = row["model_type"]
        order = row["order"]
        seasonal_order = row["seasonal_order"]

        try:
            if model_type == "Naive":
                pred = naive_forecast(y_trainval, y_test.index)
                aic = bic = np.nan
                resid = y_trainval - naive_forecast(y_trainval.iloc[:-1], y_trainval.index[1:]).reindex(y_trainval.index)
                fit = None
            elif model_type == "Seasonal_Naive":
                pred = seasonal_naive_forecast(y_trainval, y_test.index, season=12)
                aic = bic = np.nan
                resid = y_trainval - seasonal_naive_forecast(y_trainval.iloc[:12], y_trainval.index[12:], season=12).reindex(y_trainval.index)
                fit = None
            else:
                order_t = parse_tuple(order)
                seasonal_t = parse_tuple(seasonal_order)
                exog_trainval = x_trainval if model_type == "SARIMAX" else None
                exog_test = x_test if model_type == "SARIMAX" else None
                fit = fit_sarimax_model(y_trainval, order_t, seasonal_t, exog=exog_trainval)
                pred = forecast_from_fit(fit, len(y_test), y_test.index, exog=exog_test)
                aic = float(fit.aic)
                bic = float(fit.bic)
                resid = pd.Series(fit.resid).dropna()
                selected_fits[(country, model_type)] = fit

            metrics = forecast_metrics(y_test, pred)
            diag = residual_diagnostics(resid)
            selected_refit_residuals[(country, model_type)] = pd.Series(resid).dropna()

            test_rows.append({
                "country": country,
                "model_type": model_type,
                "order": order,
                "seasonal_order": seasonal_order,
                "exog_set": row.get("exog_set", None),
                "AIC_refit": aic,
                "BIC_refit": bic,
                **{f"test_{k}": v for k, v in metrics.items()},
                **diag,
                "status": "OK",
            })

            for dt, actual, forecast in zip(y_test.index, y_test.values, pred.values):
                test_forecasts.append({
                    "country": country,
                    "date": dt,
                    "model_type": model_type,
                    "order": order,
                    "seasonal_order": seasonal_order,
                    "actual": actual,
                    "forecast": forecast,
                    "error": actual - forecast,
                    "abs_error": abs(actual - forecast),
                    "squared_error": (actual - forecast) ** 2,
                })

        except Exception as e:
            test_rows.append({
                "country": country,
                "model_type": model_type,
                "order": order,
                "seasonal_order": seasonal_order,
                "exog_set": row.get("exog_set", None),
                "AIC_refit": np.nan,
                "BIC_refit": np.nan,
                "test_MAE": np.nan,
                "test_RMSE": np.nan,
                "test_sMAPE": np.nan,
                "status": f"FAILED: {type(e).__name__}: {e}",
            })

test_eval = pd.DataFrame(test_rows)
test_forecasts_df = pd.DataFrame(test_forecasts)

test_eval.to_csv(RESULTS_DIR / "03_anxiety_test_model_evaluation_enhanced.csv", index=False)
test_forecasts_df.to_csv(RESULTS_DIR / "03_anxiety_test_forecasts_enhanced.csv", index=False)

display(test_eval.sort_values(["country", "test_RMSE"]))

Test evaluation: Australia
Test evaluation: Canada
Test evaluation: Ireland
Test evaluation: New_Zealand
Test evaluation: UK
Test evaluation: US


,country,model_type,order,seasonal_order,exog_set,AIC_refit,BIC_refit,test_MAE,test_RMSE,test_sMAPE,resid_n,resid_mean,resid_std,LjungBox_p12,LjungBox_p24,ARCH_LM_p12,JarqueBera_p,status
4,Australia,Seasonal_Naive,None,None,None,NaN,NaN,3.903226,5.220926,4.512262,221,25.877828,24.840372,0.000000e+00,0.000000e+00,1.150542e-34,5.214696e-06,OK
2,Australia,SARIMA,"(1, 1, 1)","(1, 1, 1, 12)",None,1118.889277,1138.856534,7.164753,8.630130,8.280262,233,0.024589,4.133190,3.114031e-01,2.837767e-01,3.291416e-10,2.716922e-136,OK
0,Australia,ARIMA,"(3, 1, 1)","(0, 0, 0, 0)",None,1397.853829,1418.456161,10.138124,12.874957,11.760900,233,0.117569,5.261999,6.851828e-06,2.907873e-14,8.131982e-03,2.594531e-26,OK
1,Australia,Naive,None,None,None,NaN,NaN,13.935484,15.892786,15.702270,232,-35.754310,25.011426,0.000000e+00,0.000000e+00,3.133835e-39,3.372050e-06,OK
3,Australia,SARIMAX,"(1, 1, 1)","(0, 1, 1, 12)",early_covid_shock+covid_period+covid_trend,1134.396990,1161.019999,37.690423,38.805024,36.730892,233,0.106901,4.246457,1.101892e-02,1.574055e-02,8.376382e-12,1.150869e-162,OK
9,Canada,Seasonal_Naive,None,None,None,NaN,NaN,4.225806,5.064105,4.941757,221,25.122172,24.549921,0.000000e+00,0.000000e+00,6.219140e-36,4.611721e-06,OK
6,Canada,Naive,None,None,None,NaN,NaN,5.064516,5.877788,5.994683,232,-32.323276,24.629616,0.000000e+00,0.000000e+00,1.543177e-39,1.972327e-06,OK
5,Canada,ARIMA,"(2, 1, 1)","(0, 0, 0, 0)",None,1351.904717,1369.095113,7.058730,8.992276,8.209179,233,0.156318,4.930108,7.821262e-14,5.059783e-24,1.098711e-02,9.694781e-132,OK
7,Canada,SARIMA,"(1, 1, 1)","(0, 0, 1, 12)",None,1210.556372,1227.478848,9.950453,12.302622,11.266078,233,0.067127,4.330844,6.991873e-02,2.639040e-05,1.395791e-01,0.000000e+00,OK
8,Canada,SARIMAX,"(1, 1, 1)","(0, 0, 1, 12)",early_covid_shock+covid_period+covid_trend,1206.151654,1233.227614,20.084105,21.635925,21.343610,233,0.061327,4.257025,4.730056e-02,2.282879e-05,2.096875e-02,0.000000e+00,OK


## J. Diebold-Mariano Forecast Comparison

RMSE differences alone do not tell whether one forecast is statistically better than another.

This section compares each ARIMA/SARIMA/SARIMAX forecast against the best benchmark for the same country using a Diebold-Mariano style test on squared forecast errors.

Interpretation:

- Negative `DM_stat`: model has lower loss than benchmark.
- Positive `DM_stat`: model has higher loss than benchmark.
- `p < 0.05`: difference is statistically meaningful.

In [32]:
dm_rows = []

for country in COUNTRIES:
    tf = test_forecasts_df.loc[test_forecasts_df["country"].eq(country)].copy()
    country_eval = test_eval.loc[test_eval["country"].eq(country)].copy()

    benchmark_eval = country_eval.loc[country_eval["model_type"].isin(["Naive", "Seasonal_Naive"])]
    if benchmark_eval.empty:
        continue
    best_benchmark = benchmark_eval.sort_values("test_RMSE").iloc[0]["model_type"]
    benchmark_errors = tf.loc[tf["model_type"].eq(best_benchmark)].sort_values("date")["error"].values

    for model_type in ["ARIMA", "SARIMA", "SARIMAX"]:
        model_errors = tf.loc[tf["model_type"].eq(model_type)].sort_values("date")["error"].values
        if len(model_errors) != len(benchmark_errors) or len(model_errors) == 0:
            continue
        dm = diebold_mariano_test(model_errors, benchmark_errors, loss="squared", max_lag=12)
        dm_rows.append({
            "country": country,
            "model_type": model_type,
            "benchmark_model": best_benchmark,
            **dm,
            "interpretation": "negative DM means model lower loss than benchmark",
        })

dm_results = pd.DataFrame(dm_rows)
dm_results.to_csv(RESULTS_DIR / "03_anxiety_diebold_mariano_vs_best_benchmark.csv", index=False)

display(dm_results)

,country,model_type,benchmark_model,DM_stat,DM_pvalue,mean_loss_diff,n,interpretation
0,Australia,ARIMA,Seasonal_Naive,3.119852,1.809419e-03,138.506442,31,negative DM means model lower loss than benchmark
1,Australia,SARIMA,Seasonal_Naive,2.128483,3.329709e-02,47.221074,31,negative DM means model lower loss than benchmark
2,Australia,SARIMAX,Seasonal_Naive,4.387769,1.145194e-05,1478.571803,31,negative DM means model lower loss than benchmark
3,Canada,ARIMA,Seasonal_Naive,1.634839,1.020829e-01,55.215871,31,negative DM means model lower loss than benchmark
4,Canada,SARIMA,Seasonal_Naive,1.978287,4.789637e-02,125.709355,31,negative DM means model lower loss than benchmark
5,Canada,SARIMAX,Seasonal_Naive,3.198266,1.382567e-03,442.468110,31,negative DM means model lower loss than benchmark
6,Ireland,ARIMA,Seasonal_Naive,2.014160,4.399269e-02,69.444800,31,negative DM means model lower loss than benchmark
7,Ireland,SARIMA,Seasonal_Naive,1.936931,5.275383e-02,54.413597,31,negative DM means model lower loss than benchmark
8,Ireland,SARIMAX,Seasonal_Naive,4.169536,3.052200e-05,1048.406724,31,negative DM means model lower loss than benchmark
9,New_Zealand,ARIMA,Seasonal_Naive,2.873997,4.053126e-03,158.102510,31,negative DM means model lower loss than benchmark


## K. Horizon-Specific Test Errors

Benchmark models often perform strongly at short horizons. This table checks whether ARIMA/SARIMA/SARIMAX behave differently across forecast horizons.

Horizon groups:

- 1–3 months
- 4–12 months
- 13–24 months
- 25+ months

In [33]:
horizon_df = test_forecasts_df.copy()
horizon_df["horizon"] = horizon_df.groupby(["country", "model_type"]).cumcount() + 1

bins = [0, 3, 12, 24, 999]
labels = ["h01_03", "h04_12", "h13_24", "h25_plus"]
horizon_df["horizon_group"] = pd.cut(horizon_df["horizon"], bins=bins, labels=labels)

horizon_rows = []
for (country, model_type, hgroup), g in horizon_df.groupby(["country", "model_type", "horizon_group"], observed=True):
    metrics = forecast_metrics(g.set_index("date")["actual"], g.set_index("date")["forecast"])
    horizon_rows.append({
        "country": country,
        "model_type": model_type,
        "horizon_group": str(hgroup),
        "n": len(g),
        **metrics,
    })

horizon_errors = pd.DataFrame(horizon_rows)
horizon_errors.to_csv(RESULTS_DIR / "03_anxiety_horizon_specific_test_errors.csv", index=False)

display(horizon_errors.sort_values(["country", "horizon_group", "RMSE"]))

,country,model_type,horizon_group,n,MAE,RMSE,sMAPE
16,Australia,Seasonal_Naive,h01_03,3,2.666667,3.366502,3.118908
8,Australia,SARIMA,h01_03,3,2.666172,3.447805,2.945199
0,Australia,ARIMA,h01_03,3,4.020483,4.725455,4.442412
4,Australia,Naive,h01_03,3,8.000000,8.755950,8.584463
12,Australia,SARIMAX,h01_03,3,24.290163,24.368329,23.807968
...,...,...,...,...,...,...,...
107,US,Naive,h25_plus,7,3.142857,4.440077,3.423410
103,US,ARIMA,h25_plus,7,8.397760,9.095802,8.911774
119,US,Seasonal_Naive,h25_plus,7,8.714286,9.978548,9.921141
111,US,SARIMA,h25_plus,7,11.030667,12.305864,11.482907


## L. Rolling-Origin Forecast Robustness

A single validation/test split can be sensitive to the chosen dates. Rolling-origin evaluation checks whether conclusions are stable across multiple forecast origins.

This section uses the best specification selected from validation for ARIMA, SARIMA, and SARIMAX, then re-estimates it at different origins.

In [34]:
ROLLING_ORIGINS = [
    pd.Timestamp("2021-12-01"),
    pd.Timestamp("2022-12-01"),
    pd.Timestamp("2023-05-01"),
    pd.Timestamp("2023-12-01"),
]
ROLLING_HORIZON = 12

rolling_rows = []

for country in COUNTRIES:
    y = get_series(df, country, TARGET_VAR)
    x_forecast = get_exog(df, country, EXOG_FORECAST)
    country_best = best_by_class.loc[best_by_class["country"].eq(country)].copy()

    for origin in ROLLING_ORIGINS:
        y_train_roll = y.loc[y.index <= origin].dropna()
        y_future = y.loc[y.index > origin].head(ROLLING_HORIZON).dropna()
        if len(y_future) < 3:
            continue
        x_train_roll = x_forecast.loc[y_train_roll.index]
        x_future = x_forecast.loc[y_future.index]

        for _, row in country_best.iterrows():
            model_type = row["model_type"]
            try:
                if model_type == "Naive":
                    pred = naive_forecast(y_train_roll, y_future.index)
                elif model_type == "Seasonal_Naive":
                    pred = seasonal_naive_forecast(y_train_roll, y_future.index, season=12)
                else:
                    order = parse_tuple(row["order"])
                    seasonal_order = parse_tuple(row["seasonal_order"])
                    exog_train = x_train_roll if model_type == "SARIMAX" else None
                    exog_future = x_future if model_type == "SARIMAX" else None
                    fit = fit_sarimax_model(y_train_roll, order, seasonal_order, exog=exog_train)
                    pred = forecast_from_fit(fit, len(y_future), y_future.index, exog=exog_future)
                metrics = forecast_metrics(y_future, pred)
                rolling_rows.append({
                    "country": country,
                    "origin": origin,
                    "forecast_start": y_future.index.min(),
                    "forecast_end": y_future.index.max(),
                    "horizon_n": len(y_future),
                    "model_type": model_type,
                    "order": row["order"],
                    "seasonal_order": row["seasonal_order"],
                    **metrics,
                    "status": "OK",
                })
            except Exception as e:
                rolling_rows.append({
                    "country": country,
                    "origin": origin,
                    "model_type": model_type,
                    "order": row.get("order", None),
                    "seasonal_order": row.get("seasonal_order", None),
                    "MAE": np.nan,
                    "RMSE": np.nan,
                    "sMAPE": np.nan,
                    "status": f"FAILED: {type(e).__name__}: {e}",
                })

rolling_eval = pd.DataFrame(rolling_rows)
rolling_eval.to_csv(RESULTS_DIR / "03_anxiety_rolling_origin_evaluation.csv", index=False)

display(rolling_eval.sort_values(["country", "origin", "RMSE"]).groupby(["country", "origin"]).head(3))

,country,origin,forecast_start,forecast_end,horizon_n,model_type,order,seasonal_order,MAE,RMSE,sMAPE,status
2,Australia,2021-12-01,2022-01-01,2022-12-01,12,SARIMA,"(1, 1, 1)","(1, 1, 1, 12)",4.159454,5.126135,4.946766,OK
3,Australia,2021-12-01,2022-01-01,2022-12-01,12,SARIMAX,"(1, 1, 1)","(0, 1, 1, 12)",4.850713,5.775016,5.763858,OK
4,Australia,2021-12-01,2022-01-01,2022-12-01,12,Seasonal_Naive,None,None,5.416667,6.184658,6.275879,OK
7,Australia,2022-12-01,2023-01-01,2023-12-01,12,SARIMA,"(1, 1, 1)","(1, 1, 1, 12)",4.454066,5.394757,5.097350,OK
9,Australia,2022-12-01,2023-01-01,2023-12-01,12,Seasonal_Naive,None,None,4.416667,5.678908,5.171635,OK
...,...,...,...,...,...,...,...,...,...,...,...,...
110,US,2023-05-01,2023-06-01,2024-05-01,12,ARIMA,"(3, 1, 1)","(0, 0, 0, 0)",4.617395,5.446470,5.124094,OK
114,US,2023-05-01,2023-06-01,2024-05-01,12,Seasonal_Naive,None,None,5.416667,6.157651,6.167771,OK
115,US,2023-12-01,2024-01-01,2024-12-01,12,ARIMA,"(3, 1, 1)","(0, 0, 0, 0)",3.381875,4.128266,3.741317,OK
119,US,2023-12-01,2024-01-01,2024-12-01,12,Seasonal_Naive,None,None,3.333333,4.301163,3.658177,OK


## M. Full-Sample SARIMAX Intervention Model

This is the main explanatory model for the COVID effect.

It uses the full sample to estimate all intervention terms:

- `early_covid_shock`: short initial shock, March 2020–June 2020
- `covid_period`: COVID-period level shift, January 2020–May 2023
- `covid_trend`: monthly slope during COVID period
- `post_covid_period`: post-COVID level shift, June 2023 onward
- `post_covid_trend`: monthly slope after COVID

This model should be interpreted as an **association/intervention model**, not as proof of clinical mental-health causality.

In [35]:
intervention_rows = []
full_sample_sarimax_residuals = {}
full_sample_sarimax_fits = {}

for country in COUNTRIES:
    print(f"Full-sample SARIMAX intervention: {country}")
    y = get_series(df, country, TARGET_VAR).dropna()
    x = get_exog(df, country, EXOG_EXPLANATORY).loc[y.index]

    # Use the best SARIMAX order from validation when available.
    row = best_by_class.loc[(best_by_class["country"].eq(country)) & (best_by_class["model_type"].eq("SARIMAX"))]
    if row.empty:
        print(f"No SARIMAX selection for {country}; skipping.")
        continue
    row = row.iloc[0]
    order = parse_tuple(row["order"])
    seasonal_order = parse_tuple(row["seasonal_order"])

    try:
        fit = fit_sarimax_model(y, order, seasonal_order, exog=x)
        params = fit.params
        pvalues = fit.pvalues
        resid = pd.Series(fit.resid).dropna()
        full_sample_sarimax_residuals[country] = resid
        full_sample_sarimax_fits[country] = fit
        diag = residual_diagnostics(resid)

        out = {
            "country": country,
            "order": tuple_to_str(order),
            "seasonal_order": tuple_to_str(seasonal_order),
            "exog_set": "+".join(EXOG_EXPLANATORY),
            "AIC": float(fit.aic),
            "BIC": float(fit.bic),
            "converged": bool(fit.mle_retvals.get("converged", False)),
            **diag,
            "status": "OK",
        }

        for col in EXOG_EXPLANATORY:
            out[f"{col}_coef"] = float(params.get(col, np.nan))
            out[f"{col}_pvalue"] = float(pvalues.get(col, np.nan))

        intervention_rows.append(out)

    except Exception as e:
        intervention_rows.append({
            "country": country,
            "order": tuple_to_str(order),
            "seasonal_order": tuple_to_str(seasonal_order),
            "exog_set": "+".join(EXOG_EXPLANATORY),
            "AIC": np.nan,
            "BIC": np.nan,
            "converged": False,
            "status": f"FAILED: {type(e).__name__}: {e}",
        })

sarimax_intervention = pd.DataFrame(intervention_rows)
sarimax_intervention.to_csv(RESULTS_DIR / "03_anxiety_sarimax_intervention_coefficients_full_sample_enhanced.csv", index=False)
sarimax_intervention.to_csv(RESULTS_DIR / "03_final_anxiety_sarimax_intervention_table_enhanced.csv", index=False)

display(sarimax_intervention)

Full-sample SARIMAX intervention: Australia
Full-sample SARIMAX intervention: Canada
Full-sample SARIMAX intervention: Ireland
Full-sample SARIMAX intervention: New_Zealand
Full-sample SARIMAX intervention: UK
Full-sample SARIMAX intervention: US


,country,order,seasonal_order,exog_set,AIC,BIC,converged,resid_n,resid_mean,resid_std,...,early_covid_shock_coef,early_covid_shock_pvalue,covid_period_coef,covid_period_pvalue,covid_trend_coef,covid_trend_pvalue,post_covid_period_coef,post_covid_period_pvalue,post_covid_trend_coef,post_covid_trend_pvalue
0,Australia,"(1, 1, 1)","(0, 1, 1, 12)",early_covid_shock+covid_period+covid_trend+pos...,1307.593828,1342.274429,True,264,0.102630,4.183652,...,-1.089193,0.630658,-2.060245,0.594017,-0.519486,0.029926,-23.760273,0.024893,-0.961909,0.005257
1,Canada,"(1, 1, 1)","(0, 0, 1, 12)",early_covid_shock+covid_period+covid_trend+pos...,1408.280279,1443.454808,True,264,0.053696,4.376930,...,-7.876626,0.075014,1.841985,0.606859,-0.318973,0.118511,-11.146011,0.267009,-0.646557,0.001795
2,Ireland,"(2, 1, 1)","(0, 1, 1, 12)",early_covid_shock+covid_period+covid_trend+pos...,1376.663517,1414.812178,True,264,0.116920,4.694010,...,-4.680620,0.339403,-0.343463,0.962860,-0.615027,0.111888,-28.517145,0.101050,-0.844581,0.087418
3,New_Zealand,"(0, 1, 1)","(1, 0, 0, 12)",early_covid_shock+covid_period+covid_trend+pos...,1559.427179,1591.156255,True,264,0.071871,5.678170,...,0.435117,0.920681,-2.668641,0.719050,0.253800,0.662084,5.356142,0.835621,-0.893694,0.181171
4,UK,"(2, 1, 1)","(0, 0, 1, 12)",early_covid_shock+covid_period+covid_trend+pos...,1366.616931,1405.308913,True,264,0.037586,3.698107,...,-9.868752,0.000002,3.425563,0.131201,-0.314695,0.196395,-8.929540,0.395273,-0.526328,0.035090
5,US,"(1, 1, 1)","(1, 1, 1, 12)",early_covid_shock+covid_period+covid_trend+pos...,1198.476445,1236.625107,False,264,0.083259,3.440927,...,-2.439829,0.146833,0.980320,0.734895,-0.309490,0.157441,-14.386883,0.120774,-0.496986,0.049463


## N. Residual Diagnostics Summary

Diagnostics included:

- Ljung-Box p-values at lags 12 and 24: tests residual autocorrelation.
- ARCH-LM p-value: tests residual heteroskedasticity.
- Jarque-Bera p-value: tests residual normality.

For Ljung-Box:

- `p > 0.05`: residual autocorrelation is not statistically detected.
- `p ≤ 0.05`: the model still leaves autocorrelation in residuals.

In [36]:
resid_rows = []

# Selected refit residuals from train+validation
for (country, model_type), resid in selected_refit_residuals.items():
    resid_rows.append({
        "country": country,
        "model_scope": "train_validation_refit",
        "model_type": model_type,
        **residual_diagnostics(resid),
    })

# Full-sample SARIMAX intervention residuals
for country, resid in full_sample_sarimax_residuals.items():
    resid_rows.append({
        "country": country,
        "model_scope": "full_sample_intervention",
        "model_type": "SARIMAX_explanatory",
        **residual_diagnostics(resid),
    })

residual_diagnostics_df = pd.DataFrame(resid_rows)
residual_diagnostics_df.to_csv(RESULTS_DIR / "03_anxiety_residual_diagnostics_enhanced.csv", index=False)

display(residual_diagnostics_df)

,country,model_scope,model_type,resid_n,resid_mean,resid_std,LjungBox_p12,LjungBox_p24,ARCH_LM_p12,JarqueBera_p
0,Australia,train_validation_refit,ARIMA,233,0.117569,5.261999,6.851828e-06,2.907873e-14,8.131982e-03,2.594531e-26
1,Australia,train_validation_refit,Naive,232,-35.754310,25.011426,0.000000e+00,0.000000e+00,3.133835e-39,3.372050e-06
2,Australia,train_validation_refit,SARIMA,233,0.024589,4.133190,3.114031e-01,2.837767e-01,3.291416e-10,2.716922e-136
3,Australia,train_validation_refit,SARIMAX,233,0.106901,4.246457,1.101892e-02,1.574055e-02,8.376382e-12,1.150869e-162
4,Australia,train_validation_refit,Seasonal_Naive,221,25.877828,24.840372,0.000000e+00,0.000000e+00,1.150542e-34,5.214696e-06
5,Canada,train_validation_refit,ARIMA,233,0.156318,4.930108,7.821262e-14,5.059783e-24,1.098711e-02,9.694781e-132
6,Canada,train_validation_refit,Naive,232,-32.323276,24.629616,0.000000e+00,0.000000e+00,1.543177e-39,1.972327e-06
7,Canada,train_validation_refit,SARIMA,233,0.067127,4.330844,6.991873e-02,2.639040e-05,1.395791e-01,0.000000e+00
8,Canada,train_validation_refit,SARIMAX,233,0.061327,4.257025,4.730056e-02,2.282879e-05,2.096875e-02,0.000000e+00
9,Canada,train_validation_refit,Seasonal_Naive,221,25.122172,24.549921,0.000000e+00,0.000000e+00,6.219140e-36,4.611721e-06


## O. SARIMA No-COVID Counterfactual Analysis

The counterfactual model is fit only on pre-COVID data:

- Fit period: 2004-01 to 2019-12
- Forecast period: 2020-01 to 2025-12

This estimates what anxiety search behavior would have looked like if pre-COVID dynamics had continued.

### Robustness Added

To address the weakness of unconstrained SARIMA forecasts, this notebook reports three counterfactual versions:

1. `raw`: direct SARIMA forecast;
2. `capped`: raw forecast capped to the Google Trends 0–100 scale;
3. `logit`: SARIMA fitted on a logit-transformed 0–100 bounded scale.

The raw version is the main model-based forecast. Capped/logit versions are robustness checks, not replacements.

In [37]:
def scale_to_unit_interval(y, eps=1e-3):
    y = pd.Series(y).astype(float).clip(0, 100)
    return (y + eps) / (100 + 2 * eps)


def inverse_unit_interval(z, eps=1e-3):
    return pd.Series(z).astype(float) * (100 + 2 * eps) - eps


def choose_pre_covid_sarima(y_pre, candidate_specs):
    rows = []
    fits = {}
    for order, seasonal_order in candidate_specs:
        try:
            fit = fit_sarimax_model(y_pre, order, seasonal_order, exog=None)
            key = (tuple(order), tuple(seasonal_order))
            fits[key] = fit
            rows.append({
                "order": tuple_to_str(order),
                "seasonal_order": tuple_to_str(seasonal_order),
                "AIC": float(fit.aic),
                "BIC": float(fit.bic),
                "converged": bool(fit.mle_retvals.get("converged", False)),
                "status": "OK",
            })
        except Exception as e:
            rows.append({
                "order": tuple_to_str(order),
                "seasonal_order": tuple_to_str(seasonal_order),
                "AIC": np.nan,
                "BIC": np.nan,
                "converged": False,
                "status": f"FAILED: {type(e).__name__}: {e}",
            })
    table = pd.DataFrame(rows)
    ok = table.loc[table["status"].eq("OK")].dropna(subset=["AIC"])
    if ok.empty:
        raise ValueError("No pre-COVID SARIMA model converged.")
    best = ok.sort_values(["AIC", "BIC"]).iloc[0]
    key = (parse_tuple(best["order"]), parse_tuple(best["seasonal_order"]))
    return best, fits[key], table

In [38]:
counterfactual_candidate_tables = []
monthly_rows = []
phase_summary_rows = []

for country in COUNTRIES:
    print(f"Counterfactual analysis: {country}")
    y = get_series(df, country, TARGET_VAR).dropna()
    y_pre = y.loc[y.index <= PRE_COVID_END]
    y_post = y.loc[y.index >= COVID_START]
    forecast_index = y_post.index

    # Select raw SARIMA on pre-COVID period.
    try:
        best_cf, raw_fit, cf_table = choose_pre_covid_sarima(y_pre, SARIMA_SPECS)
        cf_table.insert(0, "country", country)
        counterfactual_candidate_tables.append(cf_table)

        raw_mean, raw_ci = forecast_with_ci(raw_fit, len(forecast_index), forecast_index, exog=None)
        raw_lower = raw_ci.iloc[:, 0]
        raw_upper = raw_ci.iloc[:, 1]

        capped_mean = raw_mean.clip(0, 100)
        capped_lower = raw_lower.clip(0, 100)
        capped_upper = raw_upper.clip(0, 100)

        # Logit robustness using the same selected order.
        order = parse_tuple(best_cf["order"])
        seasonal_order = parse_tuple(best_cf["seasonal_order"])
        y_unit = scale_to_unit_interval(y_pre)
        y_logit = pd.Series(logit(y_unit), index=y_pre.index)

        logit_fit = fit_sarimax_model(y_logit, order, seasonal_order, exog=None)
        logit_mean, logit_ci = forecast_with_ci(logit_fit, len(forecast_index), forecast_index, exog=None)
        logit_mean_inv = pd.Series(inverse_unit_interval(expit(logit_mean)), index=forecast_index).clip(0, 100)
        logit_lower_inv = pd.Series(inverse_unit_interval(expit(logit_ci.iloc[:, 0])), index=forecast_index).clip(0, 100)
        logit_upper_inv = pd.Series(inverse_unit_interval(expit(logit_ci.iloc[:, 1])), index=forecast_index).clip(0, 100)

        for dt in forecast_index:
            observed = float(y.loc[dt])
            phase = "COVID_period" if dt <= COVID_END else "Post_COVID"

            versions = {
                "raw": (raw_mean.loc[dt], raw_lower.loc[dt], raw_upper.loc[dt]),
                "capped": (capped_mean.loc[dt], capped_lower.loc[dt], capped_upper.loc[dt]),
                "logit": (logit_mean_inv.loc[dt], logit_lower_inv.loc[dt], logit_upper_inv.loc[dt]),
            }
            for version, (mean_v, lower_v, upper_v) in versions.items():
                monthly_rows.append({
                    "country": country,
                    "date": dt,
                    "phase": phase,
                    "counterfactual_version": version,
                    "observed": observed,
                    "counterfactual_mean": float(mean_v),
                    "counterfactual_lower_95": float(lower_v),
                    "counterfactual_upper_95": float(upper_v),
                    "gap": float(observed - mean_v),
                    "observed_above_counterfactual": int(observed > mean_v),
                    "observed_above_upper_95": int(observed > upper_v),
                    "counterfactual_exceeds_100": int(mean_v > 100),
                    "selected_order": best_cf["order"],
                    "selected_seasonal_order": best_cf["seasonal_order"],
                })

    except Exception as e:
        print(f"Counterfactual failed for {country}: {e}")

counterfactual_candidates = pd.concat(counterfactual_candidate_tables, ignore_index=True) if counterfactual_candidate_tables else pd.DataFrame()
cf_monthly = pd.DataFrame(monthly_rows)

if not cf_monthly.empty:
    for (country, version, phase), g in cf_monthly.groupby(["country", "counterfactual_version", "phase"]):
        phase_summary_rows.append({
            "country": country,
            "counterfactual_version": version,
            "phase": phase,
            "n_months": len(g),
            "mean_observed": float(g["observed"].mean()),
            "mean_counterfactual": float(g["counterfactual_mean"].mean()),
            "mean_gap": float(g["gap"].mean()),
            "cumulative_gap": float(g["gap"].sum()),
            "percent_gap_vs_counterfactual_mean": float(g["gap"].mean() / g["counterfactual_mean"].mean() * 100) if g["counterfactual_mean"].mean() != 0 else np.nan,
            "share_months_above_counterfactual": float(g["observed_above_counterfactual"].mean()),
            "share_months_above_upper_95": float(g["observed_above_upper_95"].mean()),
            "share_counterfactual_exceeds_100": float(g["counterfactual_exceeds_100"].mean()),
        })

    for (country, version), g in cf_monthly.groupby(["country", "counterfactual_version"]):
        phase_summary_rows.append({
            "country": country,
            "counterfactual_version": version,
            "phase": "Full_2020_2025",
            "n_months": len(g),
            "mean_observed": float(g["observed"].mean()),
            "mean_counterfactual": float(g["counterfactual_mean"].mean()),
            "mean_gap": float(g["gap"].mean()),
            "cumulative_gap": float(g["gap"].sum()),
            "percent_gap_vs_counterfactual_mean": float(g["gap"].mean() / g["counterfactual_mean"].mean() * 100) if g["counterfactual_mean"].mean() != 0 else np.nan,
            "share_months_above_counterfactual": float(g["observed_above_counterfactual"].mean()),
            "share_months_above_upper_95": float(g["observed_above_upper_95"].mean()),
            "share_counterfactual_exceeds_100": float(g["counterfactual_exceeds_100"].mean()),
        })

cf_gap_summary = pd.DataFrame(phase_summary_rows)

counterfactual_candidates.to_csv(RESULTS_DIR / "03_anxiety_counterfactual_candidate_models_enhanced.csv", index=False)
cf_monthly.to_csv(RESULTS_DIR / "03_anxiety_counterfactual_monthly_enhanced.csv", index=False)
cf_gap_summary.to_csv(RESULTS_DIR / "03_anxiety_counterfactual_covid_gap_summary_enhanced.csv", index=False)
cf_gap_summary.to_csv(RESULTS_DIR / "03_final_anxiety_counterfactual_effect_table_enhanced.csv", index=False)

display(cf_gap_summary.sort_values(["counterfactual_version", "country", "phase"]))

Counterfactual analysis: Australia
Counterfactual analysis: Canada
Counterfactual analysis: Ireland
Counterfactual analysis: New_Zealand
Counterfactual analysis: UK
Counterfactual analysis: US


,country,counterfactual_version,phase,n_months,mean_observed,mean_counterfactual,mean_gap,cumulative_gap,percent_gap_vs_counterfactual_mean,share_months_above_counterfactual,share_months_above_upper_95,share_counterfactual_exceeds_100
0,Australia,capped,COVID_period,41,86.707317,95.106490,-8.399173,-344.366073,-8.831335,0.024390,0.000000,0.000000
36,Australia,capped,Full_2020_2025,72,85.569444,97.119550,-11.550105,-831.607588,-11.892668,0.013889,0.000000,0.000000
1,Australia,capped,Post_COVID,31,84.064516,99.781984,-15.717468,-487.241515,-15.751810,0.000000,0.000000,0.000000
6,Canada,capped,COVID_period,41,88.829268,92.864439,-4.035170,-165.441982,-4.345227,0.341463,0.000000,0.000000
39,Canada,capped,Full_2020_2025,72,87.027778,95.590436,-8.562658,-616.511382,-8.957651,0.194444,0.000000,0.000000
7,Canada,capped,Post_COVID,31,84.645161,99.195787,-14.550626,-451.069400,-14.668593,0.000000,0.000000,0.000000
12,Ireland,capped,COVID_period,41,82.365854,89.576794,-7.210941,-295.648570,-8.050010,0.268293,0.000000,0.000000
42,Ireland,capped,Full_2020_2025,72,80.041667,93.899637,-13.857970,-997.773836,-14.758279,0.152778,0.000000,0.000000
13,Ireland,capped,Post_COVID,31,76.967742,99.616944,-22.649202,-702.125266,-22.736295,0.000000,0.000000,0.000000
18,New_Zealand,capped,COVID_period,41,89.073171,95.956304,-6.883133,-282.208459,-7.173195,0.121951,0.000000,0.000000


## P. Figures

The figures focus on the model weaknesses and how the enhanced specification addresses them:

1. Test-period forecast comparison.
2. SARIMAX intervention coefficients.
3. Counterfactual robustness: raw vs capped vs logit.
4. Residual ACF plots.

In [39]:
# 1. Test-period forecasts by country
for country in COUNTRIES:
    g = test_forecasts_df.loc[test_forecasts_df["country"].eq(country)].copy()
    if g.empty:
        continue

    fig, ax = plt.subplots(figsize=(12, 5))
    actual = g.groupby("date")["actual"].first()
    ax.plot(actual.index, actual.values, label="Actual", linewidth=2)

    # Plot selected classes only
    for model_type in ["Naive", "Seasonal_Naive", "ARIMA", "SARIMA", "SARIMAX"]:
        gg = g.loc[g["model_type"].eq(model_type)].sort_values("date")
        if not gg.empty:
            ax.plot(gg["date"], gg["forecast"], label=model_type, alpha=0.8)

    ax.axvline(TEST_START, linestyle="--", linewidth=1)
    ax.set_title(f"{country}: Test Forecasts for Anxiety Search Interest")
    ax.set_xlabel("Date")
    ax.set_ylabel("Google Trends index")
    ax.legend(ncol=3)
    fig.tight_layout()
    fig.savefig(FIGURES_DIR / f"03_anxiety_test_forecasts_{country}.png", dpi=160)
    plt.close(fig)

# 2. SARIMAX coefficient figures
if not sarimax_intervention.empty:
    coef_cols = [c for c in sarimax_intervention.columns if c.endswith("_coef")]
    for col in coef_cols:
        pcol = col.replace("_coef", "_pvalue")
        if pcol not in sarimax_intervention.columns:
            continue
        plot_df = sarimax_intervention[["country", col, pcol]].copy()
        fig, ax = plt.subplots(figsize=(10, 5))
        ax.bar(plot_df["country"], plot_df[col])
        ax.axhline(0, linewidth=1)
        ax.set_title(f"SARIMAX Intervention Coefficient: {col.replace('_coef', '')}")
        ax.set_ylabel("Coefficient")
        ax.tick_params(axis="x", rotation=30)
        for i, row in plot_df.iterrows():
            star = "*" if row[pcol] < 0.05 else ("†" if row[pcol] < 0.10 else "")
            ax.text(i, row[col], star, ha="center", va="bottom" if row[col] >= 0 else "top")
        fig.tight_layout()
        fig.savefig(FIGURES_DIR / f"03_sarimax_coef_{col.replace('_coef','')}.png", dpi=160)
        plt.close(fig)

# 3. Counterfactual robustness plots
if not cf_monthly.empty:
    for country in COUNTRIES:
        g = cf_monthly.loc[cf_monthly["country"].eq(country)].copy()
        if g.empty:
            continue
        fig, ax = plt.subplots(figsize=(12, 5))
        actual = g.groupby("date")["observed"].first()
        ax.plot(actual.index, actual.values, label="Observed", linewidth=2)

        for version in ["raw", "capped", "logit"]:
            gg = g.loc[g["counterfactual_version"].eq(version)].sort_values("date")
            if not gg.empty:
                ax.plot(gg["date"], gg["counterfactual_mean"], label=f"Counterfactual {version}", alpha=0.85)

        ax.axvspan(COVID_START, COVID_END, alpha=0.12, label="COVID period")
        ax.axvline(POST_COVID_START, linestyle="--", linewidth=1, label="Post-COVID start")
        ax.set_title(f"{country}: SARIMA No-COVID Counterfactual Robustness")
        ax.set_xlabel("Date")
        ax.set_ylabel("Google Trends index")
        ax.legend(ncol=2)
        fig.tight_layout()
        fig.savefig(FIGURES_DIR / f"03_anxiety_counterfactual_robustness_{country}.png", dpi=160)
        plt.close(fig)

# 4. Residual ACF plot for full-sample SARIMAX intervention
if full_sample_sarimax_residuals:
    n = len(full_sample_sarimax_residuals)
    fig, axes = plt.subplots(n, 1, figsize=(10, 2.8 * n))
    if n == 1:
        axes = [axes]
    for ax, (country, resid) in zip(axes, full_sample_sarimax_residuals.items()):
        plot_acf(pd.Series(resid).dropna(), lags=36, ax=ax)
        ax.set_title(f"{country}: Residual ACF — Full-Sample SARIMAX Intervention")
    fig.tight_layout()
    fig.savefig(FIGURES_DIR / "03_residual_acf_full_sample_sarimax_intervention.png", dpi=160)
    plt.close(fig)

print("Figures saved to:", FIGURES_DIR)

Figures saved to: figures


## Q. Final Summary Tables

These tables are the most important outputs for the report:

1. `final_anxiety_model_class_comparison_enhanced.csv`  
   Compares the best model from each class on the post-COVID test set.

2. `final_anxiety_sarimax_intervention_table_enhanced.csv`  
   Reports COVID-period and post-COVID coefficients from the full-sample SARIMAX intervention model.

3. `final_anxiety_counterfactual_effect_table_enhanced.csv`  
   Reports observed-vs-counterfactual gaps using raw, capped, and logit counterfactual forecasts.

4. `anxiety_diebold_mariano_vs_best_benchmark.csv`  
   Tests whether ARIMA/SARIMA/SARIMAX forecast errors are statistically different from the best benchmark.

5. `anxiety_residual_diagnostics_enhanced.csv`  
   Reports Ljung-Box, ARCH-LM, and Jarque-Bera residual diagnostics.

In [40]:
final_model_comparison = test_eval.copy()
final_model_comparison = final_model_comparison.sort_values(["country", "test_RMSE"])
final_model_comparison.to_csv(RESULTS_DIR / "03_final_anxiety_model_class_comparison_enhanced.csv", index=False)

# Report-friendly summary: best model by country on test RMSE
best_test_by_country = (
    final_model_comparison.loc[final_model_comparison["status"].eq("OK")]
    .sort_values(["country", "test_RMSE"], na_position="last")
    .groupby("country", group_keys=False)
    .head(1)
    .reset_index(drop=True)
)
best_test_by_country.to_csv(RESULTS_DIR / "03_final_anxiety_best_test_model_by_country_enhanced.csv", index=False)

# Create a compact interpretation checklist.
summary_rows = []
for country in COUNTRIES:
    best = best_test_by_country.loc[best_test_by_country["country"].eq(country)]
    if best.empty:
        continue
    best = best.iloc[0]
    int_row = sarimax_intervention.loc[sarimax_intervention["country"].eq(country)]
    if not int_row.empty:
        int_row = int_row.iloc[0]
        sig_terms = []
        for term in EXOG_EXPLANATORY:
            p = int_row.get(f"{term}_pvalue", np.nan)
            coef = int_row.get(f"{term}_coef", np.nan)
            if pd.notna(p) and p < 0.05:
                sig_terms.append(f"{term} ({coef:.3f}, p={p:.3f})")
    else:
        sig_terms = []

    summary_rows.append({
        "country": country,
        "best_test_model": best["model_type"],
        "test_RMSE": best["test_RMSE"],
        "test_sMAPE": best["test_sMAPE"],
        "significant_sarimax_terms_5pct": "; ".join(sig_terms) if sig_terms else "None",
    })

interpretation_summary = pd.DataFrame(summary_rows)
interpretation_summary.to_csv(RESULTS_DIR / "03_final_anxiety_interpretation_summary_enhanced.csv", index=False)

display(final_model_comparison)
display(best_test_by_country)
display(interpretation_summary)

print("All enhanced modeling outputs saved to:", RESULTS_DIR)

,country,model_type,order,seasonal_order,exog_set,AIC_refit,BIC_refit,test_MAE,test_RMSE,test_sMAPE,resid_n,resid_mean,resid_std,LjungBox_p12,LjungBox_p24,ARCH_LM_p12,JarqueBera_p,status
4,Australia,Seasonal_Naive,None,None,None,NaN,NaN,3.903226,5.220926,4.512262,221,25.877828,24.840372,0.000000e+00,0.000000e+00,1.150542e-34,5.214696e-06,OK
2,Australia,SARIMA,"(1, 1, 1)","(1, 1, 1, 12)",None,1118.889277,1138.856534,7.164753,8.630130,8.280262,233,0.024589,4.133190,3.114031e-01,2.837767e-01,3.291416e-10,2.716922e-136,OK
0,Australia,ARIMA,"(3, 1, 1)","(0, 0, 0, 0)",None,1397.853829,1418.456161,10.138124,12.874957,11.760900,233,0.117569,5.261999,6.851828e-06,2.907873e-14,8.131982e-03,2.594531e-26,OK
1,Australia,Naive,None,None,None,NaN,NaN,13.935484,15.892786,15.702270,232,-35.754310,25.011426,0.000000e+00,0.000000e+00,3.133835e-39,3.372050e-06,OK
3,Australia,SARIMAX,"(1, 1, 1)","(0, 1, 1, 12)",early_covid_shock+covid_period+covid_trend,1134.396990,1161.019999,37.690423,38.805024,36.730892,233,0.106901,4.246457,1.101892e-02,1.574055e-02,8.376382e-12,1.150869e-162,OK
9,Canada,Seasonal_Naive,None,None,None,NaN,NaN,4.225806,5.064105,4.941757,221,25.122172,24.549921,0.000000e+00,0.000000e+00,6.219140e-36,4.611721e-06,OK
6,Canada,Naive,None,None,None,NaN,NaN,5.064516,5.877788,5.994683,232,-32.323276,24.629616,0.000000e+00,0.000000e+00,1.543177e-39,1.972327e-06,OK
5,Canada,ARIMA,"(2, 1, 1)","(0, 0, 0, 0)",None,1351.904717,1369.095113,7.058730,8.992276,8.209179,233,0.156318,4.930108,7.821262e-14,5.059783e-24,1.098711e-02,9.694781e-132,OK
7,Canada,SARIMA,"(1, 1, 1)","(0, 0, 1, 12)",None,1210.556372,1227.478848,9.950453,12.302622,11.266078,233,0.067127,4.330844,6.991873e-02,2.639040e-05,1.395791e-01,0.000000e+00,OK
8,Canada,SARIMAX,"(1, 1, 1)","(0, 0, 1, 12)",early_covid_shock+covid_period+covid_trend,1206.151654,1233.227614,20.084105,21.635925,21.343610,233,0.061327,4.257025,4.730056e-02,2.282879e-05,2.096875e-02,0.000000e+00,OK


,country,model_type,order,seasonal_order,exog_set,AIC_refit,BIC_refit,test_MAE,test_RMSE,test_sMAPE,resid_n,resid_mean,resid_std,LjungBox_p12,LjungBox_p24,ARCH_LM_p12,JarqueBera_p,status
0,Australia,Seasonal_Naive,None,None,None,NaN,NaN,3.903226,5.220926,4.512262,221,25.877828,24.840372,0.0,0.0,1.150542e-34,0.000005,OK
1,Canada,Seasonal_Naive,None,None,None,NaN,NaN,4.225806,5.064105,4.941757,221,25.122172,24.549921,0.0,0.0,6.219140e-36,0.000005,OK
2,Ireland,Seasonal_Naive,None,None,None,NaN,NaN,4.096774,5.509523,5.252198,221,26.407240,26.976932,0.0,0.0,1.191312e-34,0.000009,OK
3,New_Zealand,Seasonal_Naive,None,None,None,NaN,NaN,6.387097,8.226629,7.401847,221,26.678733,26.641747,0.0,0.0,4.474915e-34,0.000069,OK
4,UK,Seasonal_Naive,None,None,None,NaN,NaN,3.741935,4.711003,4.422434,221,30.153846,28.847937,0.0,0.0,2.227417e-36,0.000004,OK
5,US,Naive,None,None,None,NaN,NaN,3.193548,4.299287,3.510255,232,-42.051724,23.185062,0.0,0.0,7.145778e-40,0.000004,OK


,country,best_test_model,test_RMSE,test_sMAPE,significant_sarimax_terms_5pct
0,Australia,Seasonal_Naive,5.220926,4.512262,"covid_trend (-0.519, p=0.030); post_covid_peri..."
1,Canada,Seasonal_Naive,5.064105,4.941757,"post_covid_trend (-0.647, p=0.002)"
2,Ireland,Seasonal_Naive,5.509523,5.252198,None
3,New_Zealand,Seasonal_Naive,8.226629,7.401847,None
4,UK,Seasonal_Naive,4.711003,4.422434,"early_covid_shock (-9.869, p=0.000); post_covi..."
5,US,Naive,4.299287,3.510255,"post_covid_trend (-0.497, p=0.049)"


All enhanced modeling outputs saved to: modeling_outputs


## R. Interpretation Guide

Use this guide when writing the Results section.

### Forecasting

If Naive or Seasonal Naive wins, do not treat it as model failure. It means post-COVID anxiety searches are highly persistent or seasonally regular. ARIMA/SARIMA/SARIMAX remain necessary because they explicitly model dynamics and intervention effects.

### SARIMAX Coefficients

- `early_covid_shock`: initial pandemic shock, March–June 2020.
- `covid_period`: average level shift during January 2020–May 2023.
- `covid_trend`: monthly slope within COVID period.
- `post_covid_period`: level shift after June 2023.
- `post_covid_trend`: monthly slope after June 2023.

Use coefficient signs and p-values together. A coefficient is not strong evidence unless its p-value is small and its sign is economically interpretable.

### Counterfactual

The raw SARIMA counterfactual is unconstrained and can exceed 100. Use capped and logit versions as robustness checks. If conclusions change across raw/capped/logit versions, report that the counterfactual finding is model-sensitive.

### Diagnostics

Ljung-Box p-values below 0.05 mean residual autocorrelation remains. Do not hide this. State that those country-specific findings require caution.